In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb

In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

In [4]:
SEED=42
N_FOLDS=3

air=pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")

In [5]:
air.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


In [6]:
air.describe()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
count,699635.000000,699635.000000,699635.000000,699635.000000,699635.00000,699635.00000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699343.000000
mean,349817.000000,39.001928,1353.778702,2.778722,3.15621,2.81324,3.013734,3.265644,3.364156,3.568950,3.453910,3.517404,3.472847,3.805443,3.420291,3.380401,1.176209,1.134280
std,201967.372129,13.800943,954.422566,1.280419,1.47539,1.32955,1.254014,1.300202,1.288538,1.275333,1.308157,1.238474,1.271755,1.079175,1.210604,1.277870,5.982187,5.749304
min,0.000000,7.000000,67.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,174908.500000,27.000000,631.000000,2.000000,2.00000,2.00000,2.000000,2.000000,2.000000,3.000000,2.000000,3.000000,2.000000,3.000000,3.000000,2.000000,0.000000,0.000000
50%,349817.000000,40.000000,954.000000,3.000000,3.00000,3.00000,3.000000,3.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,0.000000,0.000000
75%,524725.500000,50.000000,1814.000000,4.000000,4.00000,4.00000,4.000000,4.000000,4.000000,5.000000,5.000000,4.000000,5.000000,5.000000,4.000000,4.000000,0.000000,0.000000
max,699634.000000,85.000000,4983.000000,5.000000,5.00000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,489.000000,491.000000


In [7]:
air.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departure/Arrival time convenient  699635 non-null  int64  
 5   Ease of Online booking             699635 non-null  int64  
 6   Gate location                      699635 non-null  int64  
 7   Food and drink                     699635 non-null  int64  
 8   Online boarding                    699635 non-null  int64  
 9   Seat comfort                       699635 non-null  int64  
 10  Inflight entertainment             699635 non-null  int64  
 11  On-board service                   6996

In [8]:
air.isna().sum()

id                                     0
Age                                    0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes             292
Gender                                 0
Customer Type                          0
Type of Travel                         0
Class                                  0
satisfaction                           0
dtype: int64

In [9]:
air.duplicated().sum()

np.int64(0)

In [10]:
print(air["satisfaction"].value_counts(dropna=False))

satisfaction
False    389296
True     310339
Name: count, dtype: int64


In [11]:
print(air["satisfaction"].value_counts(normalize=True, dropna=False).mul(100).round(2))

satisfaction
False    55.64
True     44.36
Name: proportion, dtype: float64


In [12]:
# Target: 1 = satisfied, 0 = dissatisfied
y = air["satisfaction"].astype(int)

# Predictors: exclude the target and row identifier
X = air.drop(columns=["satisfaction", "id"])

print("Predictor shape:", X.shape)
print("Target counts:")
print(y.value_counts())

# Identify categorical columns and any missing values
print("\nCategorical columns:")
print(X.select_dtypes(include=["object", "string", "category"]).columns.tolist())

print("\nMissing values:")
print(X.isna().sum()[X.isna().sum() > 0])

Predictor shape: (699635, 21)
Target counts:
satisfaction
0    389296
1    310339
Name: count, dtype: int64

Categorical columns:
['Gender', 'Customer Type', 'Type of Travel', 'Class']

Missing values:
Arrival Delay in Minutes    292
dtype: int64


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)

print("\nSatisfied proportion:")
print("Training:", round(y_train.mean(), 4))
print("Validation:", round(y_valid.mean(), 4))

Training shape: (559708, 21)
Validation shape: (139927, 21)

Satisfied proportion:
Training: 0.4436
Validation: 0.4436


In [14]:
import pandas as pd

# Copy to avoid changing the original split DataFrames
X_train = X_train.copy()
X_valid = X_valid.copy()

cat_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]

for col in cat_cols:
    # Learn the available categories from training data only
    categories = X_train[col].dropna().unique()
    category_type = pd.CategoricalDtype(categories=categories)

    # Apply the same category definitions to both datasets
    X_train[col] = X_train[col].astype(category_type)
    X_valid[col] = X_valid[col].astype(category_type)

print(X_train[cat_cols].dtypes)

print("\nMissing values in training:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

Gender            category
Customer Type     category
Type of Travel    category
Class             category
dtype: object

Missing values in training:
Arrival Delay in Minutes    227
dtype: int64


In [15]:
import lightgbm as lgb

model = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols,
    eval_set=[(X_valid, y_valid)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100)
    ]
)

Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.954983
[200]	valid_0's auc: 0.956627
[300]	valid_0's auc: 0.95692
[400]	valid_0's auc: 0.957067
[500]	valid_0's auc: 0.957158
[600]	valid_0's auc: 0.957284
[700]	valid_0's auc: 0.957383
[800]	valid_0's auc: 0.957371
[900]	valid_0's auc: 0.957401
Early stopping, best iteration is:
[877]	valid_0's auc: 0.957411


LGBMClassifier(learning_rate=0.05, metric='auc', n_estimators=2000, n_jobs=-1,
               objective='binary', random_state=42, verbosity=-1)

In [16]:
from sklearn.metrics import roc_auc_score, log_loss

# Probability of satisfaction = 1
train_probs = model.predict_proba(
    X_train, num_iteration=model.best_iteration_
)[:, 1]

valid_probs = model.predict_proba(
    X_valid, num_iteration=model.best_iteration_
)[:, 1]

results = pd.DataFrame({
    "Dataset": ["Training", "Validation"],
    "ROC AUC": [
        roc_auc_score(y_train, train_probs),
        roc_auc_score(y_valid, valid_probs)
    ],
    "Log loss": [
        log_loss(y_train, train_probs),
        log_loss(y_valid, valid_probs)
    ]
})

print(results.round(6).to_string(index=False))

   Dataset  ROC AUC  Log loss
  Training 0.966005  0.216446
Validation 0.957411  0.231060


In [17]:
import numpy as np
from sklearn.metrics import log_loss

# Every passenger receives the same probability
baseline_probs = np.full(len(y_valid), y_train.mean())

baseline_loss = log_loss(y_valid, baseline_probs)
model_loss = log_loss(y_valid, valid_probs)

print(f"Baseline log loss: {baseline_loss:.6f}")
print(f"Model log loss:    {model_loss:.6f}")
print(f"Reduction:        {(1 - model_loss / baseline_loss):.2%}")

Baseline log loss: 0.686766
Model log loss:    0.231060
Reduction:        66.36%


In [18]:
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")

print("Test shape:", test.shape)
print("Unique IDs:", test["id"].is_unique)

# Compare predictor names with those used during training
test_features = set(test.columns) - {"id"}

print("Missing predictors:", set(X.columns) - test_features)
print("Unexpected columns:", test_features - set(X.columns))

print("\nMissing values:")
print(test.isna().sum()[test.isna().sum() > 0])

Test shape: (299844, 22)
Unique IDs: True
Missing predictors: set()
Unexpected columns: set()

Missing values:
Arrival Delay in Minutes    130
dtype: int64


In [19]:
# Prepare all labeled data and the test predictors
X_full = X.copy()
X_test = test[X.columns].copy()  # Keep the same feature order

# Learn category definitions from all labeled data
for col in cat_cols:
    category_type = pd.CategoricalDtype(
        categories=X_full[col].dropna().unique()
    )

    X_full[col] = X_full[col].astype(category_type)
    X_test[col] = X_test[col].astype(category_type)

# Preserve the original settings; use the selected tree count
final_params = model.get_params().copy()
final_params["n_estimators"] = model.best_iteration_

final_model = lgb.LGBMClassifier(**final_params)

final_model.fit(
    X_full,
    y,
    categorical_feature=cat_cols
)

print("Final model trained.")
print("Training rows:", len(X_full))
print("Trees:", final_model.n_estimators_)

Final model trained.
Training rows: 699635
Trees: 877


In [20]:
# Select the probability for class 1: satisfied
positive_class_index = list(final_model.classes_).index(1)
test_probs = final_model.predict_proba(X_test)[:, positive_class_index]

submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "satisfaction": test_probs
})

# Verify the submission before saving
assert list(submission.columns) == ["id", "satisfaction"]
assert len(submission) == len(test)
assert submission["id"].is_unique
assert submission["id"].equals(test["id"].reset_index(drop=True))
assert np.isfinite(submission["satisfaction"]).all()
assert submission["satisfaction"].between(0, 1).all()

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Saved submission.csv")
print("Rows:", len(submission))
print(submission.head())
print("\nProbability summary:")
print(submission["satisfaction"].describe())

Saved submission.csv
Rows: 299844
       id  satisfaction
0  699635      0.182335
1  699636      0.897809
2  699637      0.894729
3  699638      0.083687
4  699639      0.045092

Probability summary:
count    299844.000000
mean          0.443749
std           0.429926
min           0.001616
25%           0.036409
50%           0.191760
75%           0.948954
max           0.994512
Name: satisfaction, dtype: float64
